# Food Waste Estimation -- Inference Demo

Load a trained checkpoint and predict leftover weight from a before/after image pair.
Supports single-model and ensemble (all fold checkpoints averaged) inference.

**Run cells top to bottom. Do not skip the setup cell.**

## 0. Environment Setup

In [ ]:
import os, sys

try:
    import google.colab
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    PROJECT_ROOT = '/content/drive/MyDrive/ml-food-waste-estimation'  # adjust if needed
    os.chdir(PROJECT_ROOT)
else:
    if os.path.basename(os.getcwd()) == 'notebooks':
        os.chdir('..')
    PROJECT_ROOT = os.getcwd()

sys.path.insert(0, 'src')
print(f'{"Colab" if IN_COLAB else "Local"} | cwd: {os.getcwd()}')

## 1. Install Dependencies

In [ ]:
if IN_COLAB:
    import subprocess
    subprocess.run(['pip', 'install', '-r', 'requirements.txt'], check=True)
    print('Dependencies installed.')
else:
    print('Local: run "uv sync" in terminal if not done yet.')

## 2. Imports

In [ ]:
%matplotlib inline
import glob
import numpy as np
import torch
import matplotlib.pyplot as plt
from PIL import Image

from dataset import get_transforms
from model import DualStreamEfficientNet

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {DEVICE}')

## 3. Specify Image Paths

**Local**: edit `BEFORE_PATH` and `AFTER_PATH` below.  
**Colab**: the upload widget will appear automatically.

In [ ]:
if IN_COLAB:
    from google.colab import files as _files
    print('Upload the BEFORE image:')
    _up = _files.upload()
    BEFORE_PATH = list(_up.keys())[0]
    print('Upload the AFTER image:')
    _up = _files.upload()
    AFTER_PATH = list(_up.keys())[0]
else:
    # Edit these paths to point to your segmented images
    BEFORE_PATH = 'data/segmented/data_before/001_001_001_DSC_0059_bef.JPG'
    AFTER_PATH  = 'data/segmented/data_after/001_001_001_DSC_0108_aft.JPG'

print(f'Before : {BEFORE_PATH}')
print(f'After  : {AFTER_PATH}')
if not os.path.exists(BEFORE_PATH):
    print('WARNING: before image not found.')
if not os.path.exists(AFTER_PATH):
    print('WARNING: after image not found.')

## 4. Load Checkpoint

In [ ]:
CHECKPOINT_DIR = 'checkpoints'

checkpoints = sorted(glob.glob(os.path.join(CHECKPOINT_DIR, 'fold_*_best.pth')))
print(f'Found {len(checkpoints)} checkpoint(s):')
for c in checkpoints:
    print(f'  {c}')

if not checkpoints:
    raise FileNotFoundError(
        'No checkpoints found. Run training.ipynb first.'
    )

## 5. Predict

**Single model**: uses fold 1 checkpoint.  
**Ensemble**: averages predictions across all available fold checkpoints.

In [ ]:
transform = get_transforms('val')


def load_model(checkpoint_path):
    ckpt          = torch.load(checkpoint_path, map_location=DEVICE, weights_only=False)
    label_classes = ckpt.get('label_encoder_classes', [])
    num_classes   = len(label_classes) if label_classes else 34
    model         = DualStreamEfficientNet(num_classes=num_classes, pretrained=False)
    model.load_state_dict(ckpt['model_state_dict'])
    model.to(DEVICE).eval()
    return model, ckpt


def predict_single(model, ckpt, before_path, after_path):
    norm_params   = ckpt.get('normalization_params', {})
    max_weight    = norm_params.get('max_weight', 1.0)
    label_classes = ckpt.get('label_encoder_classes', [])

    before = transform(Image.open(before_path).convert('RGB')).unsqueeze(0).to(DEVICE)
    after  = transform(Image.open(after_path).convert('RGB')).unsqueeze(0).to(DEVICE)

    with torch.no_grad():
        pred_left, pred_cat = model(before, after)

    leftover_norm = float(pred_left.item())
    leftover_g    = leftover_norm * max_weight
    cat_idx       = int(pred_cat.argmax(dim=1).item())
    cat_name      = label_classes[cat_idx] if label_classes else str(cat_idx)
    confidence    = float(torch.softmax(pred_cat, dim=1).max().item())
    return leftover_norm, leftover_g, cat_name, confidence


# Single-model inference (fold 1)
model, ckpt = load_model(checkpoints[0])
norm, grams, cat, conf = predict_single(model, ckpt, BEFORE_PATH, AFTER_PATH)

print('=== Single Model (Fold 1) ===')
print(f'  Leftover (normalized) : {norm:.4f}')
print(f'  Leftover (grams)      : {grams:.1f}g')
print(f'  Food category         : {cat}')
print(f'  Confidence            : {conf:.4f}')

# Final display values -- overwritten by ensemble when available
disp_norm  = norm
disp_grams = grams
disp_label = 'Single model (fold 1)'

# Ensemble inference (all folds)
if len(checkpoints) > 1:
    all_norms, all_grams, all_confs = [], [], []
    for ckpt_path in checkpoints:
        m, c = load_model(ckpt_path)
        n, g, _, cf = predict_single(m, c, BEFORE_PATH, AFTER_PATH)
        all_norms.append(n)
        all_grams.append(g)
        all_confs.append(cf)

    disp_norm  = float(np.mean(all_norms))
    disp_grams = float(np.mean(all_grams))
    disp_label = f'Ensemble ({len(checkpoints)} folds)'

    print(f'\n=== Ensemble ({len(checkpoints)} folds) ===')
    print(f'  Leftover (normalized) : {disp_norm:.4f} +/- {np.std(all_norms):.4f}')
    print(f'  Leftover (grams)      : {disp_grams:.1f}g +/- {np.std(all_grams):.1f}g')
    print(f'  Confidence (mean)     : {np.mean(all_confs):.4f}')

## 6. Visualize

In [ ]:
before_img = Image.open(BEFORE_PATH)
after_img  = Image.open(AFTER_PATH)

fig, axes = plt.subplots(1, 2, figsize=(10, 4))
axes[0].imshow(before_img)
axes[0].set_title('Before')
axes[0].axis('off')

axes[1].imshow(after_img)
axes[1].set_title(f'After\nLeftover: {disp_grams:.1f}g ({disp_norm:.3f})')
axes[1].axis('off')

plt.suptitle(f'Food: {cat}  |  {disp_label}', fontsize=12)
plt.tight_layout()
plt.show()